In [ ]:
#!pip install transformers torch sklearn seaborn matplotlib pandas
#!pip install scipy transformers torch scikit-learn seaborn matplotlib

In [ ]:
## import libraries
import pandas as pd
import numpy as np
import random
##loss function training##
## calculate a loss function with this in the form of [0,1]

# import torch
# import torch.nn as nn
# import torch.nn.functional as F

class CosineSimilarityLoss(nn.Module):
    def __init__(self):
        super().__init__()
        return self.loss(cos_sim, labels)



import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel

from scipy.io import arff

import seaborn as sns
import matplotlib.pyplot as plt
## Tockenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
## import dataset/ look and filter texts and columns


# import test dataset
test_df = pd.read_csv("sample_data/kdd_test.csv")
print(test_df.head())


## import arff file instead of csv ###not working###

# data, meta = arff.loadarff("sample_data/KDDTest+.arff")
# print('import is working!!')

# df = pd.DataFrame(data)

# import os
# print(os.listdir('sample_data'))
# import arff

# with open('sample_data/KDDTest+.arff') as f:
#     dataset = arff.load(f)

# df = pd.DataFrame(dataset['data'])


In [ ]:
##import train data

train_df = pd.read_csv("sample_data/kdd_train.csv")
print(train_df.head())
train_df.count()


In [ ]:
print(len(train_df))
train_df = train_df.sample(
    n=5000,
    random_state=42
)
train_df.shape

In [ ]:
## creating a text column

def create_text(df):
    return df.apply(lambda row:
        f"Protocol {row['protocol_type']} Service {row['service']} Flag {row['flag']} "
        f"SrcBytes {row['src_bytes']} DstBytes {row['dst_bytes']} "
        f"Count {row['count']} SrvCount {row['srv_count']}", axis=1
    )

train_df["text"] = create_text(train_df)
test_df["text"] = create_text(test_df)

In [ ]:
##convert labels

train_labels = train_df["labels"].apply(lambda x: 0 if x == "normal" else 1).tolist()
test_labels = test_df["labels"].apply(lambda x: 0 if x == "normal" else 1).tolist()

train_texts = train_df["text"].tolist()
test_texts = test_df["text"].tolist()

In [ ]:
## create pairs function

def create_pairs(texts, labels):
    pairs = []
    pair_labels = []

    for i in range(len(texts)):
        j = random.randint(0, len(texts)-1)

        pairs.append((texts[i], texts[j]))
        pair_labels.append(int(labels[i] == labels[j]))

    return pairs, pair_labels

In [ ]:
##create_pair_function
train_pairs, train_pair_labels = create_pairs(train_texts, train_labels)
test_pairs, test_pair_labels = create_pairs(test_texts, test_labels)

In [ ]:
##test ccreate_pair function
print(train_pairs[0])
print(train_pair_labels[0])



In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
## defining encxode function

def encode(text):
    return tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

In [ ]:
##sbertmodel defination
from transformers import AutoModel
import torch.nn as nn
import torch

In [ ]:


class SBERTModel(nn.Module):
    def __init__(self, model_name="bert-base-uncased"):
        super(SBERTModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sentence_embedding = self.mean_pooling(outputs, attention_mask)
        return sentence_embedding

In [ ]:

##class distribution pie chart to show normal traffic and attack trafffic

attack_counts = train_df["labels"].value_counts()

plt.figure(figsize=(6,6))
plt.pie(
    attack_counts,
    labels=attack_counts.index,
    autopct='%1.1f%%'
)

plt.title("Distribution of Network Traffic Classes")
plt.show()

In [ ]:
##protocol distrubution bar chart

train_df["protocol_type"].value_counts().plot(kind='bar')

plt.title("Protocol Distribution")
plt.xlabel("Protocol")
plt.ylabel("Count")

plt.show()

In [ ]:
##bar chart for service frequency(what kind of attack is happening)

train_df["service"].value_counts().head(10).plot(kind='bar')

plt.title("Top 10 Network Services")
plt.xlabel("Service")
plt.ylabel("Frequency")

plt.show()

In [ ]:
## attack vs normal traffic count

binary_labels = train_df["labels"].apply(
    lambda x: "Normal" if x == "normal" else "Attack"
)

binary_labels.value_counts().plot(kind='bar')

plt.title("Attack vs Normal Traffic")
plt.show()

In [ ]:
# # ##confusion matrix

# sns.heatmap(cm, annot=True, fmt='d')

# plt.title("Confusion Matrix")
# plt.xlabel("Predicted")
# plt.ylabel("Actual")

# plt.show()

In [ ]:
##define cosine similarity function
class CosineSimilarityLoss(nn.Module):

    def __init__(self):
        super().__init__()

        self.loss = nn.MSELoss()

    def forward(self, emb1, emb2, labels):

        cos_sim = F.cosine_similarity(emb1, emb2)

        # normalize from [-1,1] → [0,1]
        cos_sim = (cos_sim + 1) / 2

        return self.loss(
            cos_sim,
            labels.float()
        )

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SBERTModel().to(device)

criterion = CosineSimilarityLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-5
)

In [ ]:
## try with first 1000 data points first

train_pairs = train_pairs[:1000]
train_pair_labels = train_pair_labels[:1000]

In [ ]:
## fed the training data to the model

model.train()

for epoch in range(8):

    total_loss = 0

    for (text1, text2), label in zip(
        train_pairs,
        train_pair_labels
    ):

        enc1 = encode(text1)
        enc2 = encode(text2)

        emb1 = model(
            enc1['input_ids'].to(device),
            enc1['attention_mask'].to(device)
        )

        emb2 = model(
            enc2['input_ids'].to(device),
            enc2['attention_mask'].to(device)
        )

        loss = criterion(
            emb1,
            emb2,
            torch.tensor([label]).to(device)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
# #saving trained model
# torch.save(model.state_dict(), "sbert_model.pt")
# # Then immediately copy to Google Drive so it survives session end
# from google.colab import drive
# drive.mount('/content/drive')

# torch.save(model.state_dict(), "/content/drive/MyDrive/sbert_model.pt")

In [ ]:
## model evaluation and variable defination

model.eval()


In [ ]:
y_true = []
y_pred = []
cos_values = []

### 300 data points used
test_pairs = test_pairs[:300]
test_pair_labels = test_pair_labels[:300]

In [ ]:
####evaluation loop
with torch.no_grad():

    for (text1, text2), label in zip(
        test_pairs,
        test_pair_labels
    ):

        enc1 = encode(text1)
        enc2 = encode(text2)
        emb1 = model(
            enc1['input_ids'].to(device),
            enc1['attention_mask'].to(device)
        )

        emb2 = model(
            enc2['input_ids'].to(device),
            enc2['attention_mask'].to(device)
        )

        cos_sim = F.cosine_similarity(emb1, emb2)

        cos_sim = (cos_sim + 1) / 2

        pred = (cos_sim > 0.5).int()

        y_true.append(label)
        y_pred.append(pred.item())

        cos_values.append(cos_sim.item())

In [ ]:
###confusion matrix generation(graph first)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
####evaluation loop
with torch.no_grad():

    for (text1, text2), label in zip(
        test_pairs,
        test_pair_labels
    ):

        enc1 = encode(text1)
        enc2 = encode(text2)
        emb1 = model(
            enc1['input_ids'].to(device),
            enc1['attention_mask'].to(device)
        )

        emb2 = model(
            enc2['input_ids'].to(device),
            enc2['attention_mask'].to(device)
        )

        cos_sim = F.cosine_similarity(emb1, emb2)

        cos_sim = (cos_sim + 1) / 2

        pred = (cos_sim > 0.5).int()

        y_true.append(label)
        y_pred.append(pred.item())

        cos_values.append(cos_sim.item())
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
#cosine similarity

import seaborn as sns
###confusion matrix generation(graph first)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
####evaluation loop
with torch.no_grad():

    for (text1, text2), label in zip(
        test_pairs,
        test_pair_labels
    ):

        enc1 = encode(text1)
        enc2 = encode(text2)
        emb1 = model(
            enc1['input_ids'].to(device),
            enc1['attention_mask'].to(device)
        )

        emb2 = model(
            enc2['input_ids'].to(device),
            enc2['attention_mask'].to(device)
        )

        cos_sim = F.cosine_similarity(emb1, emb2)

        cos_sim = (cos_sim + 1) / 2

        pred = (cos_sim > 0.5).int()

        y_true.append(label)
        y_pred.append(pred.item())

        cos_values.append(cos_sim.item())
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()
sims = [F.cosine_similarity(
    model(encode(t1)['input_ids'].to(device), encode(t1)['attention_mask'].to(device)),
    model(encode(t2)['input_ids'].to(device), encode(t2)['attention_mask'].to(device))
).item() for (t1,t2) in train_pairs[:200]]

sns.histplot(sims, bins=30)
plt.title("Cosine Similarity Distribution"); plt.show()

In [ ]:
import matplotlib.pyplot as plt
import collections

counts = collections.Counter(train_pair_labels)
labels = ['Dissimilar (0)', 'Similar (1)']
sizes = [counts[0], counts[1]]
colors = ['#FF6B6B', '#4ECDC4']

fig, ax = plt.subplots(fig#cosine similarity

import seaborn as sns
###confusion matrix generation(graph first)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
####evaluation loop
with torch.no_grad():

    for (text1, text2), label in zip(
        test_pairs,
        test_pair_labels
    ):

        enc1 = encode(text1)
        enc2 = encode(text2)
        emb1 = model(
            enc1['input_ids'].to(device),
            enc1['attention_mask'].to(device)
        )

        emb2 = model(
            enc2['input_ids'].to(device),
            enc2['attention_mask'].to(device)
        )

        cos_sim = F.cosine_similarity(emb1, emb2)

        cos_sim = (cos_sim + 1) / 2

        pred = (cos_sim > 0.5).int()

        y_true.append(label)
        y_pred.append(pred.item())

        cos_values.append(cos_sim.item())
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()
sims = [F.cosine_similarity(
    model(encode(t1)['input_ids'].to(device), encode(t1)['attention_mask'].to(device)),
    model(encode(t2)['input_ids'].to(device), encode(t2)['attention_mask'].to(device))
).item() for (t1,t2) in train_pairs[:200]]

sns.histplot(sims, bins=30)
plt.title("Cosine Similarity Distribution"); plt.show()size=(7, 7))
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.5)  # makes it a donut
)
autotexts[0].set_fontsize(13)
autotexts[1].set_fontsize(13)
ax.set_title("Label Distribution", fontsize=15, fontweight='bold')
plt.show()

In [ ]:
import numpy as np
from scipy.ndimage import gaussian_filter1d

smoothed = gaussian_filter1d(epoch_losses, sigma=0.8)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, 9), epoch_losses, 'o', color='#4ECDC4', alpha=0.5, label='Raw loss')
ax.plot(range(1, 9), smoothed, '-', color='#2C7873', linewidth=2.5, label='Smoothed')
ax.fill_between(range(1, 9), smoothed, alpha=0.15, color='#4ECDC4')
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Training Loss Curve", fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
###figure3(false positive reduction rate)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)

print("False Positive Rate:", fpr)

In [ ]:
###figure 4: performance comparision of proposed ai model with traditional method

systems = ["Traditional SIEM", "Proposed AI Model"]

false_positives = [120, fp]

plt.figure(figsize=(7,5))

plt.bar(systems, false_positives)

plt.title("False Positive Reduction")
plt.ylabel("Number of False Positives")

plt.show()
